In [11]:
import pandas as pd
from pathlib import Path


# =============================================================================
# Config
# =============================================================================

RESULTS_CSV = Path(
    "/Users/aumona/Projects/RF-GAP-Python/experiments/results/20260413_022933_ablation/kernel_method/kernel_method_results.csv"
)

DATASET = "airlines"
MODEL_TYPE = "rf"

# Choose which proximity methods to include, in the order you want in the table
PROX_TO_KEEP = ["gap", "oob", "kerf", "original"]


# =============================================================================
# Helpers
# =============================================================================

def fmt_pm(mean, std, decimals=3):
    return rf"${mean:.{decimals}f} \pm {std:.{decimals}f}$"

def train_size_label(n):
    return f"{int(n):,}"


# =============================================================================
# Load and filter
# =============================================================================

df = pd.read_csv(RESULTS_CSV)

df = df[
    (df["dataset"] == DATASET)
    & (df["model_type"] == MODEL_TYPE)
    & (df["kernel_method"].isin(PROX_TO_KEEP))
    & (df["status"] == "ok")
].copy()

if len(df) == 0:
    raise ValueError("No rows left after filtering.")


# =============================================================================
# Forest accuracy: same forest repeated across proximity methods
# Keep one copy per seed / train size
# =============================================================================

forest_df = (
    df[["dataset", "seed", "n_train_subset", "forest_test_acc"]]
    .drop_duplicates()
    .copy()
)

forest_summary = (
    forest_df.groupby("n_train_subset", as_index=False)
    .agg(
        forest_mean=("forest_test_acc", "mean"),
        forest_std=("forest_test_acc", "std"),
    )
    .sort_values("n_train_subset")
    .reset_index(drop=True)
)

forest_summary["forest_std"] = forest_summary["forest_std"].fillna(0.0)
forest_summary["Forest"] = [
    fmt_pm(m, s)
    for m, s in zip(forest_summary["forest_mean"], forest_summary["forest_std"])
]


# =============================================================================
# Kernel prediction accuracy by proximity method
# =============================================================================

kernel_summary = (
    df.groupby(["kernel_method", "n_train_subset"], as_index=False)
    .agg(
        mean_acc=("kernel_predict_test_acc", "mean"),
        std_acc=("kernel_predict_test_acc", "std"),
    )
    .sort_values(["kernel_method", "n_train_subset"])
    .reset_index(drop=True)
)

kernel_summary["std_acc"] = kernel_summary["std_acc"].fillna(0.0)
kernel_summary["acc_fmt"] = [
    fmt_pm(m, s)
    for m, s in zip(kernel_summary["mean_acc"], kernel_summary["std_acc"])
]


# =============================================================================
# Build final wide table
# =============================================================================

table_df = forest_summary[["n_train_subset", "Forest"]].copy()

for prox in PROX_TO_KEEP:
    sub = (
        kernel_summary[kernel_summary["kernel_method"] == prox][["n_train_subset", "acc_fmt"]]
        .rename(columns={"acc_fmt": prox.upper()})
        .copy()
    )
    table_df = table_df.merge(sub, on="n_train_subset", how="outer")

table_df = table_df.sort_values("n_train_subset").reset_index(drop=True)
table_df["Train size"] = table_df["n_train_subset"].map(train_size_label)

final_cols = ["Train size", "Forest"] + [prox.upper() for prox in PROX_TO_KEEP]
table_df = table_df[final_cols]

print(table_df.to_string(index=False))

# =============================================================================
# Manual LaTeX output (no jinja2 needed), wrapped in table*
# =============================================================================

latex_cols = final_cols

lines = []
lines.append(r"\begin{table*}[t]")
lines.append(r"\centering")
lines.append(r"\caption{Test accuracy of the forest predictor and kernel-weighted predictors across training sizes on " + DATASET.capitalize() + r".}")
lines.append(r"\label{tab:" + DATASET + r"_kernel_predict_acc}")
lines.append(r"\begin{tabular}{" + "l" + "c" * (len(latex_cols) - 1) + "}")
lines.append(r"\toprule")
lines.append(" & ".join(latex_cols) + r" \\")
lines.append(r"\midrule")

for _, row in table_df.iterrows():
    vals = [str(row[col]) for col in latex_cols]
    lines.append(" & ".join(vals) + r" \\")

lines.append(r"\bottomrule")
lines.append(r"\end{tabular}")
lines.append(r"\end{table*}")

latex_table = "\n".join(lines)
print("\n" + latex_table)

Train size            Forest               GAP               OOB              KERF          ORIGINAL
    16,384 $0.629 \pm 0.002$ $0.630 \pm 0.002$ $0.645 \pm 0.003$ $0.633 \pm 0.002$ $0.645 \pm 0.002$
    32,768 $0.630 \pm 0.003$ $0.630 \pm 0.003$ $0.649 \pm 0.003$ $0.633 \pm 0.003$ $0.649 \pm 0.003$
    65,536 $0.629 \pm 0.003$ $0.628 \pm 0.003$ $0.651 \pm 0.002$ $0.631 \pm 0.003$ $0.650 \pm 0.003$
   131,072 $0.624 \pm 0.001$ $0.624 \pm 0.001$ $0.652 \pm 0.002$ $0.627 \pm 0.002$ $0.648 \pm 0.002$
   262,144 $0.618 \pm 0.002$ $0.618 \pm 0.003$ $0.651 \pm 0.002$ $0.622 \pm 0.002$ $0.646 \pm 0.003$
   485,444 $0.619 \pm 0.002$ $0.619 \pm 0.002$ $0.645 \pm 0.002$ $0.625 \pm 0.002$ $0.642 \pm 0.003$

\begin{table*}[t]
\centering
\caption{Test accuracy of the forest predictor and kernel-weighted predictors across training sizes on Airlines.}
\label{tab:airlines_kernel_predict_acc}
\begin{tabular}{lccccc}
\toprule
Train size & Forest & GAP & OOB & KERF & ORIGINAL \\
\midrule
16,384 & $0.62